In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
HERE = %pwd
sys.path.append(os.path.dirname(HERE))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
    
import numpy as np
import pandas as pd
import copy
import pickle
import time
import collections
from tqdm import tqdm

In [2]:
from src import utils
rng = utils.set_seed()

dir_parent = utils.dir_parent
version_exp = utils.version_exp
dir_workspace = f"{dir_parent}/research/TFCSR"

In [3]:
# https://grouplens.org/datasets/movielens/1m/
dir_load = f"{dir_parent}/received_data/ml-1m"
dir_data = f"{dir_workspace}/preprocessed_data/{version_exp}/MovieLens"
os.makedirs(dir_data, exist_ok=True)

# parameters
n_positive = 2
n_candidate = 50 - n_positive
n_user = 500

# Load data

In [4]:
# user master

## https://files.grouplens.org/datasets/movielens/ml-1m-README.txt
d_age = {
    1: "Under 18",
    18: "18-24",
    25: "25-34",
    35: "35-44",
    45: "45-49",
    50: "50-55",
    56: "56+"
}

d_occupation = {
    0: "other or not specified",
    1: "academic/educator",
    2: "artist",
    3: "clerical/admin",
    4: "college/grad student",
    5: "customer service",
    6: "doctor/health care",
    7: "executive/managerial",
    8: "farmer",
    9: "homemaker",
    10: "K-12 student",
    11: "lawyer",
    12: "programmer",
    13: "retired",
    14: "sales/marketing",
    15: "scientist",
    16: "self-employed",
    17: "technician/engineer",
    18: "tradesman/craftsman",
    19: "unemployed",
    20: "writer"
}

df_ = pd.read_csv(f'{dir_load}/users.dat', sep="::", engine="python", names=["userID", "Gender", "Age", "Occupation", "Zip-code"])
df_ = df_.drop("Zip-code", axis=1)
df_["Age"] = df_["Age"].map(d_age)
df_ = df_[df_["Occupation"] != 0]  # remove "other or not specified"
df_["Occupation"] = df_["Occupation"].map(d_occupation)
df_ = df_.set_index("userID")
df_ = df_.T.add_prefix("U").T
df_.columns = [utils.change_col(a) for a in df_.columns]
df_users = df_.copy()
df_users.head()

,gender,age,occupation
userID,,,
U1,F,Under 18,K-12 student
U2,M,56+,self-employed
U3,M,25-34,scientist
U4,M,45-49,executive/managerial
U5,M,25-34,writer


In [5]:
# item master
df_ = pd.read_csv(f'{dir_load}/movies.dat', sep="::", engine="python", names=["itemID", "Title", "Genres"], encoding='ISO-8859-1', on_bad_lines='skip')
df_["Genres"] = df_["Genres"].apply(lambda s : ", ".join(s.split("|")))
df_ = df_.set_index("itemID")
df_ = df_.T.add_prefix("I").T
df_.columns = [utils.change_col(a) for a in df_.columns]
df_items = df_.copy()
df_items.head()

,title,genres
itemID,,
I1,Toy Story (1995),"Animation, Children's, Comedy"
I2,Jumanji (1995),"Adventure, Children's, Fantasy"
I3,Grumpier Old Men (1995),"Comedy, Romance"
I4,Waiting to Exhale (1995),"Comedy, Drama"
I5,Father of the Bride Part II (1995),Comedy


In [6]:
# transaction records
df_ = pd.read_csv(f'{dir_load}/ratings.dat', sep="::", engine="python", names=["userID", "itemID", "Rating", "Timestamp"]).rename(
    columns={"Rating" : "rating", "Timestamp" : "timestamp"}
)
df_["userID"] = df_["userID"].apply(lambda s : f"U{s}")
df_["itemID"] = df_["itemID"].apply(lambda s : f"I{s}")

## restrict to users registered in the user master
set_users = set(df_users.index.values)
df_ = df_[[i in set_users for i in df_["userID"].values]]  

df_ = df_.drop_duplicates(subset=["userID", "itemID"], keep='first')
df_records = df_.copy()
df_records.head()

,userID,itemID,rating,timestamp
0,U1,I1193,5,978300760
1,U1,I661,3,978302109
2,U1,I914,3,978301968
3,U1,I3408,4,978300275
4,U1,I2355,5,978824291


# Create preprocessed data

In [7]:
# frequent items up to rank 10000
s = df_records["itemID"].value_counts()
s = s.sort_values(ascending=False).iloc[:1000]
set_items = set(s.index)

gb = df_records.groupby("userID")
users = df_records["userID"].unique()
k = -1*n_positive


def _items_user(user, gb):
    df_ = gb.get_group(user).sort_values(by="timestamp", ascending=True)
    df_ = df_[df_["rating"] > 3]
    items_user = df_["itemID"].values    
    return items_user

def _candidate_items(items_user, set_items, k, n_candidate):
    items_pos = items_user[k:]  # latest k items
    items_neg = rng.choice(list(set_items - set(items_user)), size=n_candidate, replace=False)      
    d_ = {
        "candidates_positive" : ", ".join(items_pos),
        "candidates_negative" : ", ".join(items_neg)
    }
    return copy.deepcopy(d_)

dd_data = dict()

### User profile

In [8]:
# shuffle
users = rng.choice(users, size=len(users), replace=False)

d_data = dict()
idx = 0
for user in tqdm(users):
    items_user = _items_user(user, gb)
    
    if len(items_user) >= n_positive:
        try:
            # candidate items
            d_ = _candidate_items(items_user, set_items, k, n_candidate)
            
            # profile
            d_["profile"] = df_users.loc[user].to_dict()
            d_data[user] = copy.deepcopy(d_)
            idx += 1
        except:
            pass
    
    if idx == n_user:
        print(user, d_data[user])
        break

dd_data["profile"] = copy.deepcopy(d_data)

  9%|████████▌                                                                                   | 499/5329 [00:00<00:06, 790.55it/s]

U2850 {'candidates_positive': 'I3168, I648', 'candidates_negative': 'I3770, I3844, I272, I1747, I2396, I410, I2687, I1193, I2336, I1277, I2671, I2080, I1094, I1721, I3072, I196, I3098, I720, I3301, I173, I2253, I1590, I162, I1589, I2413, I2722, I1327, I1262, I1674, I1242, I1608, I185, I2707, I1347, I2501, I3363, I317, I1958, I2826, I3435, I1834, I3461, I1093, I1466, I524, I2716, I2076, I1982', 'profile': {'gender': 'M', 'age': '18-24', 'occupation': 'writer'}}


### User history

In [9]:
N_icl = [1, 3, 5, 10]
for n_icl in N_icl:
    # shuffle
    users = rng.choice(users, size=len(users), replace=False)
    
    d_data = dict()
    idx = 0
    for user in tqdm(users):
        items_user = _items_user(user, gb)
        
        if len(items_user) >= n_icl + n_positive:
            try:
                # candidate items
                d_ = _candidate_items(items_user, set_items, k, n_candidate)
                
                # user history (except latest k items)
                d_["history"] = {i+1 : item for i,item in enumerate(items_user[:k][::-1][:n_icl])} 
                d_data[user] = copy.deepcopy(d_)
                idx += 1
            except:
                pass
    
        if idx == n_user:
            print(user, d_data[user])
            break

    dd_data[f"{n_icl}-sample"] = copy.deepcopy(d_data)

  9%|████████▌                                                                                  | 500/5329 [00:00<00:03, 1318.90it/s]


U3831 {'candidates_positive': 'I2413, I2245', 'candidates_negative': 'I2728, I3168, I1284, I1030, I3397, I2779, I144, I3489, I317, I2416, I2541, I1208, I3633, I2006, I3258, I3809, I546, I3911, I2160, I1297, I2421, I376, I2395, I1193, I3927, I2398, I1179, I357, I1466, I1378, I1639, I2396, I2953, I2003, I648, I1027, I3264, I3468, I1100, I1060, I2124, I2671, I3152, I198, I104, I3528, I1569, I532', 'history': {1: 'I2336'}}


  9%|████████▌                                                                                  | 500/5329 [00:00<00:03, 1269.20it/s]


U3814 {'candidates_positive': 'I145, I1587', 'candidates_negative': 'I2294, I432, I2009, I2803, I3916, I2407, I1968, I2413, I1175, I2160, I3698, I1097, I333, I858, I265, I2248, I163, I3301, I2000, I1388, I1954, I25, I161, I1199, I3683, I47, I520, I1088, I3334, I1784, I2600, I2405, I595, I2622, I364, I150, I3261, I720, I3479, I1267, I1719, I1950, I2788, I910, I784, I3697, I2369, I1028', 'history': {1: 'I3773', 2: 'I3640', 3: 'I3812'}}


  9%|████████▌                                                                                  | 503/5329 [00:00<00:03, 1289.91it/s]


U1062 {'candidates_positive': 'I3755, I3863', 'candidates_negative': 'I648, I1060, I3398, I410, I2739, I3545, I2143, I2827, I16, I2141, I2950, I172, I1357, I1095, I1955, I3198, I1285, I3267, I2020, I2398, I62, I153, I2004, I1358, I1029, I2746, I161, I799, I1411, I2792, I303, I3785, I151, I1882, I2539, I420, I2795, I2406, I45, I2942, I2231, I1566, I3258, I236, I318, I2336, I1207, I223', 'history': {1: 'I1409', 2: 'I3501', 3: 'I2109', 4: 'I2469', 5: 'I2870'}}


 10%|████████▋                                                                                  | 510/5329 [00:00<00:03, 1313.41it/s]

U2519 {'candidates_positive': 'I364, I2173', 'candidates_negative': 'I1172, I105, I1254, I2194, I2455, I922, I1485, I7, I1952, I24, I150, I3717, I1589, I2502, I2600, I2391, I457, I3068, I104, I3952, I1215, I441, I2770, I1625, I541, I3871, I1380, I994, I555, I2915, I1080, I3791, I34, I2324, I3253, I3438, I2302, I3735, I2, I784, I3168, I2112, I2110, I1257, I3083, I2997, I1378, I3361', 'history': {1: 'I1220', 2: 'I3545', 3: 'I1947', 4: 'I1028', 5: 'I1282', 6: 'I2080', 7: 'I1288', 8: 'I595', 9: 'I899', 10: 'I2946'}}


### Save

In [10]:
d_item = dict()
for user_type, d_data in dd_data.items():
    with open(f"{dir_data}/records_{user_type}.pickle", 'wb') as f:
        pickle.dump(d_data, f)
    
    # items that used in the main experiments
    try:
        items_history = np.unique(np.concatenate([list(d["history"].values()) for d in d_data.values()]))
    except:
        items_history = []
    
    items_pos = np.unique(np.concatenate([d["candidates_positive"].split(", ") for d in d_data.values()]))
    items_neg = np.unique(np.concatenate([d["candidates_negative"].split(", ") for d in d_data.values()]))
    items_candidates = np.unique(np.concatenate([items_pos, items_neg]))
            
    d_item[user_type] = {
        "history" : items_history,
        "candidates" : items_candidates
    }

for item_type in ["history", "candidates"]:
    items_ = np.unique(np.concatenate([d[item_type] for d in d_item.values()]))
    df_i = df_items.loc[items_]
    df_i.to_csv(f"{dir_data}/items_{item_type}.csv")
    display(df_i.head())
    print(f"{len(df_i)} items for {item_type} record.")

,title,genres
itemID,,
I1,Toy Story (1995),"Animation, Children's, Comedy"
I10,GoldenEye (1995),"Action, Adventure, Thriller"
I1003,Extreme Measures (1996),"Drama, Thriller"
I1007,"Apple Dumpling Gang, The (1975)","Children's, Comedy, Western"
I1008,"Davy Crockett, King of the Wild Frontier (1955)",Western


1985 items for history record.


,title,genres
itemID,,
I1,Toy Story (1995),"Animation, Children's, Comedy"
I10,GoldenEye (1995),"Action, Adventure, Thriller"
I1006,"Chamber, The (1996)",Drama
I101,Bottle Rocket (1996),Comedy
I1012,Old Yeller (1957),"Children's, Drama"


1693 items for candidates record.
